# Модель Росса (резервирование и ремонт)

Модель представляет собой классический пример системы массового обслуживания
с конечной популяцией, резервом и ремонтом.

## Постановка задачи

- В системе находятся N идентичных машин, которые постоянно работают и могут выходить из строя.
- S машин находятся в резерве и готовы немедленно заменить любую отказавшую.
- R ремонтных устройств (ремонтников), которые могут одновременно ремонтировать R машин.
- Когда работающая машина ломается:
  - Немедленно берётся одна резервная машина (если есть) и запускается в работу.
  - Сломанная машина отправляется в ремонт.
  - Если резерва нет, система падает (crash).
- После ремонта машина пополняет пул резервных.

In [ ]:
# Подключение модулей

using DrWatson
@quickactivate "project"
include(srcdir("Ross.jl"))
using .Ross
using Plots

# Параметры модели

- `N = 10` — количество основных машин
- `S = 3` — количество резервных машин
- `num_repairers = 1` — количество ремонтников
- `λ = 100.0` — средняя наработка на отказ (часов)
- `μ = 1.0` — среднее время ремонта (часов)

In [ ]:
N = 10
S = 3
num_repairers = 1

println("=== Модель Росса ===")
println("N=$N, S=$S, ремонтников=$num_repairers")

# Запуск симуляции

result = run_ross_simulation(N=N, S=S, num_repairers=num_repairers, seed=123)

# Вывод статистики

print_stats(result)

# Построение и сохранение графиков

График показывает изменение количества резервных машин во времени.
Когда резервные машины заканчиваются (спад до 0), система падает.

In [ ]:
p = plot_ross_results(result)
if p !== nothing
    savefig(plotsdir("ross_results.png"))
    println("\nГрафик сохранён в plots/ross_results.png")
    display(p)
end

# Исследование влияния количества ремонтников

Проведём серию экспериментов с разным числом ремонтников (1, 2, 3)
и сравним время до падения системы.

In [ ]:
println("\n=== Исследование влияния количества ремонтников ===")
for r in [1, 2, 3]
    res = run_ross_simulation(N=10, S=3, num_repairers=r, seed=123)
    println("Ремонтников: $r, Время падения: $(round(res.stop_time, digits=2)), Результат: $(res.msg)")
end

# Выводы

- Увеличение числа ремонтников значительно увеличивает время жизни системы
- При одном ремонтнике резерв быстро истощается
- При трёх ремонтниках система может работать неограниченно долго
- Модель демонстрирует важность резервирования и достаточного числа ремонтного персонала